In [2]:
import polars as pl
import numpy as np

# 2020

## Создаем список торгов

In [5]:
df = pl.read_excel(source='correct_data/correct_data_2020.xlsx')
df = df.select(['ИНН заказчика', 'Реестровый номер публикации'])
df = df.rename({'Реестровый номер публикации':'pub_num'})
df = df.unique().sort('pub_num')
df

ИНН заказчика,pub_num
str,str
"""0276090428""","""0101100001120000001"""
"""2014030545""","""0101100009420000001"""
"""2014030545""","""0101100009420000002"""
"""0261014135""","""0101200002320000001"""
"""0261014135""","""0101200002320000002"""
…,…
"""0276143694""","""3961555"""
"""0276143694""","""3963183"""
"""0276143694""","""3963353"""


In [9]:
df.write_excel()

## Проверяем сколько закупок по каждому фз

In [181]:
d = pl.read_excel('zakgov/data_for_parsing_2020.xlsx')
d

ИНН заказчика,pub_num
str,str
"""0276090428""","""0101100001120000001"""
"""2014030545""","""0101100009420000001"""
"""2014030545""","""0101100009420000002"""
"""0261014135""","""0101200002320000001"""
"""0261014135""","""0101200002320000002"""
…,…
"""0276143694""","""3961555"""
"""0276143694""","""3963183"""
"""0276143694""","""3963353"""


In [182]:
print(f"Закупки по 44/94 ФЗ: {d.filter(pl.col('pub_num').str.len_chars()==19).shape[0]}")
print(f"Закупки с других площадок: {d.filter(pl.col('pub_num').str.len_chars()==7).shape[0]}")
print(f"Закупки по 223 ФЗ: {d.filter(pl.col('pub_num').str.len_chars()==11).shape[0]}")

Закупки по 44/94 ФЗ: 12974
Закупки с других площадок: 1799
Закупки по 223 ФЗ: 48860


### Нашли тендеры с несколькими участниками

In [183]:
g = d.filter(pl.col('pub_num').str.len_chars()==19)
g = g.group_by(pl.col('pub_num')).agg(pl.col('ИНН заказчика').count().alias('n'))
dif_prices = g.filter(pl.col('n')>1)
dif_prices.filter(pl.col('pub_num')=='0131300045220000003')

pub_num,n
str,u32
"""0131300045220000003""",13


In [184]:
dif_prices.select(pl.col('n').sum())

n
u32
350


## Парсинг 2020

In [168]:
import json

with open('zakgov/result_zakgov_2020.json', 'r') as f:
    data = json.load(f)


ozoik = pl.from_dicts(data)
ozoik

pub_num,price,oz,oik,no_info
str,str,str,str,i64
"""0101100001120000001""",""" 2 000…",""" 20 000,00 РОССИЙСКИЙ …",""" 100…",0
"""0101200002320000001""",""" 16 50…",""" 165 000,00 РОССИЙСКИЙ…",""" 825…",0
"""0101200002320000002""",""" 4 000…",""" 40 000,00 РОССИЙСКИЙ …",""" …",0
"""0101200009320000001""",""" 22 68…",null,""" 2 2…",0
"""0101100009420000002""",""" 634 1…",null,""" 31 …",0
…,…,…,…,…
"""32514569675""",""" …",null,null,0
"""32514571035""",""" …",null,null,0
"""32514570297""",""" …",null,null,0


In [169]:
ozoik = ozoik.drop('no_info')
ozoik = ozoik.select('pub_num', 'price')

In [132]:
ozoik.write_excel()

## Делаем нормальные значения цены

In [170]:
ozoik = ozoik.with_columns(
                            pl.col("price")
                            .str.replace_all(r"[^\d,]", "")  
                            .str.replace(",", ".")            
                            .cast(pl.Float64)                 
                            .alias("price_float")
                           )

In [171]:
ozoik = ozoik.select('pub_num', 'price_float')
ozoik = ozoik.rename({'pub_num': 'Реестровый номер публикации', 'price_float':'Стоимость(руб.) Заказчик'})

## Соединяем с основным датафреймом

In [225]:
data = pl.read_excel(source='correct_data/correct_data_2020.xlsx')
print(data.shape)
data.head(2)

(109900, 26)


Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
2,"""ГБУЗ АО ""ГКБ №3""""","""3018005693""",49120.0,"""0325500000120000008""","""20-23018005693302301001-0002-0…","""[ОКПД2 01.47] Птица сельскохоз…","""Астраханская область""","""Астрахань""",2020-01-17 11:42:46,2020-01-27 09:00:00,2020-01-17 13:42:45,2020-01-28 00:00:00,"""Полянский Владимир Владимирови…","""301501912801""","""Победитель""","""Допущен""",0.397394,"""Торговая процедура""",null,null,null,null,0.05,0,0
1,"""ФГБУ ""ВИМС""""","""7706433263""",420000.0,"""32009380421""",null,"""[ОКПД2 29.20] Кузова (корпуса)…","""Иркутская область""","""Иркутская область""",2020-08-05 13:07:54,2020-08-05 13:20:00,2020-08-05 00:00:00,2020-08-05 00:00:00,"""ООО ""ТК ""ЕВА""""","""3810065555""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0


In [226]:
df_noprice = data.filter(pl.col('Стоимость(руб.) Заказчик').is_null())
df_noprice.shape

(36823, 26)

In [227]:
noprice_list = df_noprice.select(pl.col('Реестровый номер публикации').unique()).to_numpy().flatten()
diffprice_list = dif_prices.select(pl.col('pub_num').unique()).to_numpy().flatten()
print(noprice_list.shape, diffprice_list.shape)
print(np.sum(np.isin(noprice_list, diffprice_list)))

(11986,) (25,)
0


In [228]:
df_noprice = df_noprice.join(other=ozoik.unique(), on='Реестровый номер публикации', how='left')
print(df_noprice.shape)
df_noprice.head(2)

(36823, 27)


Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее,Стоимость(руб.) Заказчик_right
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64,f64
2,"""ООО ""БСК""""","""0273056757""",null,"""3607048""",null,"""[ОКПД2 28.13] Насосы и компрес…","""Республика Башкортостан""","""Уфа""",2024-03-15 14:28:57,2024-03-26 19:29:25,2024-03-15 14:28:57,2024-05-03 00:00:00,"""ООО ""БЭК""""","""0278114628""","""Победитель""","""Неизвестно""",0.244663,"""Торговая процедура""",null,null,null,null,null,0,0,null
2,"""ООО ""САМЭСК""""","""6319231042""",null,"""32211463136""",null,"""[ОКПД2 41.20] Здания и работы …","""Самарская область""","""Самарская область""",2022-06-10 11:41:54,2022-06-21 09:30:00,2022-06-14 00:00:00,2022-06-21 00:00:00,"""АО ""САМАРА-ВЭМ""""","""6316060656""",null,"""Допущен""",0.040555,"""Торговая процедура""",null,null,null,null,null,0,0,330258.6


In [229]:
names = df_noprice.columns
names =  names[:3]+names[26:27]+names[4:26]
df_noprice = df_noprice.select(names)
df_noprice = df_noprice.rename({'Стоимость(руб.) Заказчик_right':'Стоимость(руб.) Заказчик'})
print(df_noprice.shape)
df_noprice.head(2)

(36823, 26)


Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
2,"""ООО ""БСК""""","""0273056757""",null,"""3607048""",null,"""[ОКПД2 28.13] Насосы и компрес…","""Республика Башкортостан""","""Уфа""",2024-03-15 14:28:57,2024-03-26 19:29:25,2024-03-15 14:28:57,2024-05-03 00:00:00,"""ООО ""БЭК""""","""0278114628""","""Победитель""","""Неизвестно""",0.244663,"""Торговая процедура""",null,null,null,null,null,0,0
2,"""ООО ""САМЭСК""""","""6319231042""",330258.6,"""32211463136""",null,"""[ОКПД2 41.20] Здания и работы …","""Самарская область""","""Самарская область""",2022-06-10 11:41:54,2022-06-21 09:30:00,2022-06-14 00:00:00,2022-06-21 00:00:00,"""АО ""САМАРА-ВЭМ""""","""6316060656""",null,"""Допущен""",0.040555,"""Торговая процедура""",null,null,null,null,null,0,0


In [230]:
newdf = pl.concat([data.filter(pl.col('Стоимость(руб.) Заказчик').is_not_null()), df_noprice])
newdf

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
2,"""ГБУЗ АО ""ГКБ №3""""","""3018005693""",49120.0,"""0325500000120000008""","""20-23018005693302301001-0002-0…","""[ОКПД2 01.47] Птица сельскохоз…","""Астраханская область""","""Астрахань""",2020-01-17 11:42:46,2020-01-27 09:00:00,2020-01-17 13:42:45,2020-01-28 00:00:00,"""Полянский Владимир Владимирови…","""301501912801""","""Победитель""","""Допущен""",0.397394,"""Торговая процедура""",null,null,null,null,0.05,0,0
1,"""ФГБУ ""ВИМС""""","""7706433263""",420000.0,"""32009380421""",null,"""[ОКПД2 29.20] Кузова (корпуса)…","""Иркутская область""","""Иркутская область""",2020-08-05 13:07:54,2020-08-05 13:20:00,2020-08-05 00:00:00,2020-08-05 00:00:00,"""ООО ""ТК ""ЕВА""""","""3810065555""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0
1,"""ИСПМ РАН""","""7728021249""",366354.41,"""32110151407""",null,"""[ОКПД2 23.19] Стекло прочее, в…","""Москва""","""Москва""",2021-04-01 15:24:24,2021-04-01 16:00:00,2021-04-01 00:00:00,2021-04-01 00:00:00,"""ООО ""ЛАБОРАТОРНАЯ ТЕХНИКА""""","""7719636091""","""Победитель""","""Неизвестно""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0
1,"""ФГАОУ ДПО ""АКАДЕМИЯ МИНПРОСВЕЩ…","""7718084063""",495702.0,"""32312121334""",null,"""[ОКПД2 43.29] Работы строитель…","""Москва""","""Москва""",2023-02-15 11:48:56,2023-02-15 12:00:00,2023-02-15 00:00:00,2023-02-15 00:00:00,"""Пурис Алина Айратовна""","""026826054589""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0
2,"""ФКУ ИК-4 УФСИН РОССИИ ПО РЕСПУ…","""1651020150""",215864.0,"""0311100004120000004""","""20-11651020150165101001-0009-0…","""[ОКПД2 22.22] Изделия пластмас…","""Республика Татарстан (Татарста…","""Республика Татарстан (Татарста…",2020-01-22 15:24:30,2020-01-30 08:00:00,2020-01-22 15:44:30,2020-01-30 00:00:00,"""Кожакин Владислав Олегович""","""165057384785""",null,"""Допущен""",0.175,"""Торговая процедура""",null,null,null,10793.2,0.05,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",4.8129e6,"""32009305437""",null,"""[ОКПД2 71.12] Услуги в области…","""Москва""","""Москва""",2020-07-09 09:01:44,2020-07-29 10:30:00,2020-07-09 00:00:00,2020-07-31 10:30:00,"""ООО ""ИНСТИТУТ ВНИИЖЕЛЕЗОБЕТОН""""","""7720367541""",null,"""Допущен""",0.0,"""Торговая процедура""",null,null,null,null,null,0,0
2,"""МАОУ СОШ № 38""","""3906045856""",768572.0,"""32110152379""",null,"""[ОКПД2 46.90] Услуги по неспец…","""Калининградская область""","""Калининградская область""",2021-04-01 18:18:20,2021-04-02 12:00:00,2021-04-01 00:00:00,2021-04-02 00:00:00,"""ООО ""ПРОФИНТЕГРО""""","""3906384697""",null,"""Допущен""",-0.033996,"""Торговая процедура""",null,null,null,null,null,0,0
2,"""АО ""ОЗК""""","""7708632345""",7.30785e8,"""32211237428""",null,"""[ОКПД2 52.10] Услуги по склади…","""Москва""","""Москва""",2022-03-18 18:20:39,2022-04-18 10:00:00,2022-03-18 00:00:00,2022-04-29 13:30:00,"""ООО ""КАНСКОЕ ХПП""""","""2450022978""",null,"""Допущен""",0.029126,"""Торговая процедура""",null,5000.0,null,null,null,0,0


## Заполняем пропуски

In [234]:
new = newdf.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_not_null() &
                                pl.col('Обеспечение заявки (руб.)').is_not_null() &
                                pl.col('Обеспечение заявки, %').is_null())
                            .then(pl.col('Обеспечение заявки (руб.)')/pl.col('Стоимость(руб.) Заказчик'))
                            .otherwise(pl.col('Обеспечение заявки, %'))
                            .alias('Обеспечение заявки, %'))
new = new.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_not_null() &
                                pl.col('Обеспечение заявки (руб.)').is_null() &
                                pl.col('Обеспечение заявки, %').is_not_null())
                            .then(pl.col('Обеспечение заявки, %')*pl.col('Стоимость(руб.) Заказчик'))
                            .otherwise(pl.col('Обеспечение заявки (руб.)'))
                            .alias('Обеспечение заявки (руб.)'))

new = new.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_not_null() &
                                pl.col('Обеспечение контракта (руб.)').is_not_null() &
                                pl.col('Обеспечение контракта, %').is_null())
                            .then(pl.col('Обеспечение контракта (руб.)')/pl.col('Стоимость(руб.) Заказчик'))
                            .otherwise(pl.col('Обеспечение контракта, %'))
                            .alias('Обеспечение контракта, %'))
new = new.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_not_null() &
                                pl.col('Обеспечение контракта (руб.)').is_null() &
                                pl.col('Обеспечение контракта, %').is_not_null())
                            .then(pl.col('Обеспечение контракта, %')*pl.col('Стоимость(руб.) Заказчик'))
                            .otherwise(pl.col('Обеспечение контракта (руб.)'))
                            .alias('Обеспечение контракта (руб.)'))

In [236]:
new.shape

(109900, 26)

In [239]:
new.write_excel()

# 2015

## Создаем список торгов

In [7]:
df = pl.read_excel(source='correct_data/correct_data_2015.xlsx')
df = df.select(['ИНН заказчика', 'Реестровый номер публикации'])
df = df.rename({'Реестровый номер публикации':'pub_num'})
df = df.unique().sort('pub_num')
df

Could not determine dtype for column 5, falling back to string


ИНН заказчика,pub_num
str,str
"""7451432207""","""0321718002DP"""
"""7451432207""","""0321718003DP"""
"""7710305634""","""1000111"""
"""7710305634""","""1009211"""
"""7710305634""","""1030788"""
…,…
"""7710305634""","""966553"""
"""7710305634""","""970998"""
"""7710305634""","""973130"""


In [8]:
df = df.filter(pl.col('pub_num').str.len_chars()==11)
df 

ИНН заказчика,pub_num
str,str
"""6670480610""","""31806697369"""
"""5074070474""","""31806713098"""
"""5074070474""","""31806713320"""
"""5074070474""","""31806713373"""
"""5074070474""","""31806722871"""
…,…
"""5321062368""","""32009107963"""
"""5321062368""","""32009108289"""
"""5321062368""","""32009108424"""


### Нашли тендеры с несколькими участниками

In [10]:
g = df.group_by(pl.col('pub_num')).agg(pl.col('ИНН заказчика').count().alias('n'))
dif_prices = g.filter(pl.col('n')>1)
dif_prices.shape

(121, 2)

In [17]:
dif_prices.head(3)

pub_num,n
str,u32
"""31806957852""",4
"""31807154691""",2
"""31807168641""",2


Убираем их

In [11]:
for_sure = g.filter(pl.col('n')==1)
for_sure.shape

(158244, 2)

In [15]:
forsure_df = df.join(for_sure, on='pub_num', how='left')
forsure_df = forsure_df.filter(pl.col('n').is_not_null()).drop('n')
forsure_df.head(2)

ИНН заказчика,pub_num
str,str
"""6670480610""","""31806697369"""
"""5074070474""","""31806713098"""


In [16]:
forsure_df.write_excel()

## Парсинг 2015

In [28]:
import json

with open('zakgov/result_zakgov_2015.json', 'r') as f:
    data = json.load(f)


ozoik = pl.from_dicts(data)
ozoik.shape

(147742, 5)

In [29]:
ozoik.null_count()

pub_num,price,oz,oik,no_info
u32,u32,u32,u32,u32
0,0,147742,147742,0


In [30]:
ozoik = ozoik.drop('no_info', 'oz', 'oik')
print(ozoik.shape)
ozoik.head(2)

(147742, 2)


pub_num,price
str,str
"""31806697369""",""" …"
"""31806713320""",""" …"


## Делаем нормальные значения цены

In [31]:
ozoik = ozoik.with_columns(
                            pl.col("price")
                            .str.replace_all(r"[^\d,]", "")  
                            .str.replace(",", ".")            
                            .cast(pl.Float64)                 
                            .alias("price")
                           )
ozoik = ozoik.rename({'pub_num': 'Реестровый номер публикации', 'price':'Стоимость(руб.) Заказчик'})
ozoik.head(2)

Реестровый номер публикации,Стоимость(руб.) Заказчик
str,f64
"""31806697369""",1.1564e7
"""31806713320""",964033.33


## Соединяем с основным датафреймом

In [34]:
data = pl.read_excel(source='correct_data/correct_data_2015.xlsx')
print(data.shape)
data.head(2)

Could not determine dtype for column 5, falling back to string


(382178, 22)


Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,i64,i64
1,"""АНО ""ДФИКММ ЧО""""","""7451432207""",1.4641e7,"""0321718002DP""",null,"""[ОКПД2 27.20] Батареи и аккуму…","""Челябинская область""","""Челябинск""",2018-12-14 12:35:50,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""ТЕХНО-ЦЕНТР""""","""7453181622""","""Победитель""","""Допущен""",0.048235,"""Торговая процедура""","""Иной способ""",0,0
2,"""АНО ""ДФИКММ ЧО""""","""7451432207""",null,"""0321718003DP""",null,"""[ОКПД2 28.25] Оборудование про…","""Челябинская область""","""Челябинск""",2018-12-14 12:39:28,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""МАСТЕРВЕНТ""""","""7451246578""",null,"""Не допущен""",0.171902,"""Торговая процедура""",null,0,0


In [48]:
data.filter(pl.col('Стоимость(руб.) Заказчик').is_null()).shape

(319522, 22)

In [37]:
new = data.join(other=ozoik.unique(), on='Реестровый номер публикации', how='left')
print(new.shape)
new.head(2)

(382178, 23)


Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,РНП сейчас,РНП ранее,Стоимость(руб.) Заказчик_right
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,i64,i64,f64
1,"""АНО ""ДФИКММ ЧО""""","""7451432207""",1.4641e7,"""0321718002DP""",null,"""[ОКПД2 27.20] Батареи и аккуму…","""Челябинская область""","""Челябинск""",2018-12-14 12:35:50,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""ТЕХНО-ЦЕНТР""""","""7453181622""","""Победитель""","""Допущен""",0.048235,"""Торговая процедура""","""Иной способ""",0,0,null
2,"""АНО ""ДФИКММ ЧО""""","""7451432207""",null,"""0321718003DP""",null,"""[ОКПД2 28.25] Оборудование про…","""Челябинская область""","""Челябинск""",2018-12-14 12:39:28,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""МАСТЕРВЕНТ""""","""7451246578""",null,"""Не допущен""",0.171902,"""Торговая процедура""",null,0,0,null


In [42]:
new = new.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_null() & pl.col('Стоимость(руб.) Заказчик_right').is_not_null())
                       .then(pl.col('Стоимость(руб.) Заказчик_right'))
                       .when(pl.col('Стоимость(руб.) Заказчик').is_not_null() & pl.col('Стоимость(руб.) Заказчик_right').is_null())
                       .then(pl.col('Стоимость(руб.) Заказчик'))
                       .when(pl.col('Стоимость(руб.) Заказчик').is_not_null() & pl.col('Стоимость(руб.) Заказчик_right').is_not_null())
                       .then(pl.col('Стоимость(руб.) Заказчик'))
                       .alias('price'))
print(new.shape)

(382178, 24)


In [49]:
new.columns

['Уровень',
 'Заказчик',
 'ИНН заказчика',
 'Стоимость(руб.) Заказчик',
 'Реестровый номер публикации',
 'Идентификационный код закупки',
 'Сфера деятельности',
 'Регион поставки',
 'Город поставки',
 'Дата публикации',
 'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
 'Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ',
 'Дата окончания проведения торгов',
 'Поставщик',
 'ИНН поставщика',
 'Победитель',
 'Статус допуска',
 'Снижение на торгах,%',
 'Форма публикации',
 'Тип торгов',
 'РНП сейчас',
 'РНП ранее',
 'Стоимость(руб.) Заказчик_right',
 'price']

In [52]:
name = new.columns
name = name[0:3]+name[23:24]+name[4:22]
new = new.select(name)
print(new.shape)
new.head(2)

(382178, 22)


Уровень,Заказчик,ИНН заказчика,price,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,i64,i64
1,"""АНО ""ДФИКММ ЧО""""","""7451432207""",1.4641e7,"""0321718002DP""",null,"""[ОКПД2 27.20] Батареи и аккуму…","""Челябинская область""","""Челябинск""",2018-12-14 12:35:50,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""ТЕХНО-ЦЕНТР""""","""7453181622""","""Победитель""","""Допущен""",0.048235,"""Торговая процедура""","""Иной способ""",0,0
2,"""АНО ""ДФИКММ ЧО""""","""7451432207""",null,"""0321718003DP""",null,"""[ОКПД2 28.25] Оборудование про…","""Челябинская область""","""Челябинск""",2018-12-14 12:39:28,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""МАСТЕРВЕНТ""""","""7451246578""",null,"""Не допущен""",0.171902,"""Торговая процедура""",null,0,0


In [54]:
new = new.with_columns(pl.lit(None, dtype=pl.Float64).alias('Обеспечение заявки (руб.)'),
                      pl.lit(None, dtype=pl.Float64).alias('Обеспечение заявки, %'),
                      pl.lit(None, dtype=pl.Float64).alias('Обеспечение контракта (руб.)'),
                      pl.lit(None, dtype=pl.Float64).alias('Обеспечение контракта, %'))
new.head(1)

Уровень,Заказчик,ИНН заказчика,price,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,РНП сейчас,РНП ранее,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %"
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,i64,i64,f64,f64,f64,f64
1,"""АНО ""ДФИКММ ЧО""""","""7451432207""",1.4641e7,"""0321718002DP""",null,"""[ОКПД2 27.20] Батареи и аккуму…","""Челябинская область""","""Челябинск""",2018-12-14 12:35:50,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""ТЕХНО-ЦЕНТР""""","""7453181622""","""Победитель""","""Допущен""",0.048235,"""Торговая процедура""","""Иной способ""",0,0,null,null,null,null


In [55]:
new.columns

['Уровень',
 'Заказчик',
 'ИНН заказчика',
 'price',
 'Реестровый номер публикации',
 'Идентификационный код закупки',
 'Сфера деятельности',
 'Регион поставки',
 'Город поставки',
 'Дата публикации',
 'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
 'Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ',
 'Дата окончания проведения торгов',
 'Поставщик',
 'ИНН поставщика',
 'Победитель',
 'Статус допуска',
 'Снижение на торгах,%',
 'Форма публикации',
 'Тип торгов',
 'РНП сейчас',
 'РНП ранее',
 'Обеспечение заявки (руб.)',
 'Обеспечение заявки, %',
 'Обеспечение контракта (руб.)',
 'Обеспечение контракта, %']

In [62]:
names = new.columns
names = names[:20] + names[22:] + names[20:22]
new = new.select(names)
print(new.shape)
new.head(2)

(382178, 26)


Уровень,Заказчик,ИНН заказчика,price,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
1,"""АНО ""ДФИКММ ЧО""""","""7451432207""",1.4641e7,"""0321718002DP""",null,"""[ОКПД2 27.20] Батареи и аккуму…","""Челябинская область""","""Челябинск""",2018-12-14 12:35:50,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""ТЕХНО-ЦЕНТР""""","""7453181622""","""Победитель""","""Допущен""",0.048235,"""Торговая процедура""","""Иной способ""",null,null,null,null,0,0
2,"""АНО ""ДФИКММ ЧО""""","""7451432207""",null,"""0321718003DP""",null,"""[ОКПД2 28.25] Оборудование про…","""Челябинская область""","""Челябинск""",2018-12-14 12:39:28,2018-12-19 13:00:00,2018-12-14 00:00:00,2018-12-20 16:00:00,"""ООО ""МАСТЕРВЕНТ""""","""7451246578""",null,"""Не допущен""",0.171902,"""Торговая процедура""",null,null,null,null,null,0,0


In [ ]:
new = new.rename({'price':'Стоимость(руб.) Заказчик'})

In [71]:
new.write_excel()

In [73]:
new.null_count()

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,39818,0,382178,0,0,0,0,0,0,0,0,0,211401,0,0,0,321261,382178,382178,382178,382178,0,0


In [72]:
print(new.shape, data.shape)

(382178, 26) (382178, 22)


In [74]:
data.null_count()

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,РНП сейчас,РНП ранее
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,319522,0,382178,0,0,0,0,0,0,0,0,0,211401,0,0,0,321261,0,0


In [76]:
data.shape[0]-39818

342360

нашли много данных по ценам)

## Делаем нормальные значения oz и oik

In [61]:
ozoik = ozoik.rename({'oz':'обеспечение заявки', 
                      'oik':'обеспечение исполнения контракта',  
                      'pub_num':'Реестровый номер публикации'})

In [62]:
# исправляем oz
ozoik = ozoik.with_columns(pl.col("обеспечение заявки")
                                   .str.extract(r"([\d\xa0,]+)")
                                   .str.replace_all("\xa0", "")
                                   .str.replace(",", ".")
                                   .cast(pl.Float64)
                                   .alias("обеспечение заявки"))

In [63]:
# сиправляем oik
ozoik = ozoik.with_columns(
                    pl.col("обеспечение исполнения контракта")
                    .str.extract(r'([\d\xa0,]+)\s*₽')
                    .str.replace_all(r'[\xa0,]', '')
                    .cast(pl.Float64)
                    .alias("numeric_value"),
                
                    pl.col("обеспечение исполнения контракта")
                    .str.extract(r'\(?(\d+,\d+|\d+)\s*%\)?')
                    .str.replace(',', '.')
                    .cast(pl.Float64)
                    .alias("percentage_value"))

#ozoik = ozoik.with_columns(pl.when((pl.col('numeric_value').is_not_null()) & (pl.col('percentage_value').is_not_null())).then(pl.lit(None)).otherwise(pl.col('numeric_value')).alias('numeric_value'))

ozoik = ozoik.rename({'numeric_value':'обеспечение исполнения контракта, руб', 'percentage_value': 'обеспечение исполнения контракта, %'})
ozoik = ozoik.with_columns(pl.col('обеспечение исполнения контракта, %')/100)
ozoik

Реестровый номер публикации,обеспечение заявки,обеспечение исполнения контракта,no_info,"обеспечение исполнения контракта, руб","обеспечение исполнения контракта, %"
str,f64,str,i64,f64,f64
"""0101100001120000001""",20000.0,""" 100…",0,1e7,0.05
"""0101100009420000002""",null,""" 31 …",0,3.170953e6,0.05
"""0101200002320000002""",40000.0,""" …",0,null,0.05
"""0101100009420000001""",null,""" 49 …",0,4.952138e6,0.05
"""0101200008020000001""",null,""" …",0,null,0.05
…,…,…,…,…,…
"""2071500000120000002""",null,""" 28 …",0,2.802586e6,null
"""2074700000120000001""",14400.0,""" …",0,null,0.05
"""2084700000120000001""",null,""" …",0,null,0.05


In [64]:
ozoik = ozoik.drop('no_info')

## Соединяем с основным датафреймом

In [65]:
data = pl.read_excel(source='correct_data/correct_data_2020.xlsx')
data.shape

(109900, 26)

In [66]:
data = data.join(other=ozoik.unique('Реестровый номер публикации'), how='left', on=['Реестровый номер публикации'])
data

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее,обеспечение заявки,обеспечение исполнения контракта,"обеспечение исполнения контракта, руб","обеспечение исполнения контракта, %"
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64,f64,str,f64,f64
2,"""ГБУЗ АО ""ГКБ №3""""","""3018005693""",49120.0,"""0325500000120000008""","""20-23018005693302301001-0002-0…","""[ОКПД2 01.47] Птица сельскохоз…","""Астраханская область""","""Астрахань""",2020-01-17 11:42:46,2020-01-27 09:00:00,2020-01-17 13:42:45,2020-01-28 00:00:00,"""Полянский Владимир Владимирови…","""301501912801""","""Победитель""","""Допущен""",0.397394,"""Торговая процедура""",null,null,null,null,0.05,0,0,null,""" …",null,0.05
1,"""ФГБУ ""ВИМС""""","""7706433263""",420000.0,"""32009380421""",null,"""[ОКПД2 29.20] Кузова (корпуса)…","""Иркутская область""","""Иркутская область""",2020-08-05 13:07:54,2020-08-05 13:20:00,2020-08-05 00:00:00,2020-08-05 00:00:00,"""ООО ""ТК ""ЕВА""""","""3810065555""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0,null,null,null,null
2,"""ООО ""БСК""""","""0273056757""",null,"""3607048""",null,"""[ОКПД2 28.13] Насосы и компрес…","""Республика Башкортостан""","""Уфа""",2024-03-15 14:28:57,2024-03-26 19:29:25,2024-03-15 14:28:57,2024-05-03 00:00:00,"""ООО ""БЭК""""","""0278114628""","""Победитель""","""Неизвестно""",0.244663,"""Торговая процедура""",null,null,null,null,null,0,0,null,null,null,null
1,"""ИСПМ РАН""","""7728021249""",366354.41,"""32110151407""",null,"""[ОКПД2 23.19] Стекло прочее, в…","""Москва""","""Москва""",2021-04-01 15:24:24,2021-04-01 16:00:00,2021-04-01 00:00:00,2021-04-01 00:00:00,"""ООО ""ЛАБОРАТОРНАЯ ТЕХНИКА""""","""7719636091""","""Победитель""","""Неизвестно""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0,null,null,null,null
1,"""ФГАОУ ДПО ""АКАДЕМИЯ МИНПРОСВЕЩ…","""7718084063""",495702.0,"""32312121334""",null,"""[ОКПД2 43.29] Работы строитель…","""Москва""","""Москва""",2023-02-15 11:48:56,2023-02-15 12:00:00,2023-02-15 00:00:00,2023-02-15 00:00:00,"""Пурис Алина Айратовна""","""026826054589""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Закупка у единственного постав…",null,null,null,null,0,0,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,"""ГКБСМП""","""3906040840""",96283.83,"""0335300053220000003""","""20-23906040840390601001-0045-0…","""[ОКПД2 21.20] Препараты лекарс…","""Калининградская область""","""Калининград""",2020-01-23 13:17:30,2020-01-31 10:00:00,2020-01-23 15:26:10,2020-02-03 00:00:00,"""ООО ""МЕДИАЛАБ""""","""3906306265""","""Победитель""","""Допущен""",0.298761,"""Торговая процедура""",null,null,null,null,0.05,0,0,null,""" …",null,0.05
1,"""КГКУ ХОРСКИЙ СРЦ""","""2713012203""",519830.0,"""0122200002520000033""","""20-22713012203271301001-0017-0…","""[ОКПД2 19.20] Нефтепродукты, […","""Хабаровский край""","""им Лазо район""",2020-01-20 12:00:10,2020-01-29 07:00:00,2020-01-20 16:23:21,2020-01-30 00:00:00,"""АО ""ННК-ХАБАРОВСКНЕФТЕПРОДУКТ""""","""2700000105""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Аукцион электронный""",null,null,25991.5,null,0,0,null,""" 25 …",2.59915e6,0.05
2,"""КОМИТЕТ ЖКХ, ТИС""","""2902012008""",5.922485e6,"""01243000127200

In [89]:
check_oik = data.with_columns(pl.when(pl.col('Обеспечение контракта (руб.)').is_not_null()).then(1).otherwise(0).alias('oldr'))
check_oik = check_oik.with_columns(pl.when(pl.col('Обеспечение контракта, %').is_not_null()).then(1).otherwise(0).alias('oldpr'))
check_oik = check_oik.with_columns((pl.col('oldr')+pl.col('oldpr')).alias('old'))
check_oik = check_oik.with_columns(pl.when(pl.col('old')>0).then(1).otherwise(0).alias('old'))

check_oik = check_oik.with_columns(pl.when(pl.col('обеспечение исполнения контракта, руб').is_not_null()).then(1).otherwise(0).alias('newr'))
check_oik = check_oik.with_columns(pl.when(pl.col('обеспечение исполнения контракта, %').is_not_null()).then(1).otherwise(0).alias('newpr'))
check_oik = check_oik.with_columns((pl.col('newr')+pl.col('newpr')).alias('new'))
check_oik = check_oik.with_columns(pl.when(pl.col('new')>0).then(1).otherwise(0).alias('new'))
check_oik = check_oik.drop(['oldr','oldpr', 'newr', 'newpr'])
check_oik.filter(pl.col('old')!=pl.col('new'))

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее,обеспечение заявки,обеспечение исполнения контракта,"обеспечение исполнения контракта, руб","обеспечение исполнения контракта, %",old,new
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64,f64,str,f64,f64,i32,i32


короче, ничего нового не нашли(

In [91]:
check_oz = data.with_columns(pl.when(pl.col('Обеспечение заявки (руб.)').is_not_null()).then(1).otherwise(0).alias('oldr'))
check_oz = check_oz.with_columns(pl.when(pl.col('Обеспечение заявки, %').is_not_null()).then(1).otherwise(0).alias('oldpr'))
check_oz = check_oz.with_columns((pl.col('oldr')+pl.col('oldpr')).alias('old'))
check_oz = check_oz.with_columns(pl.when(pl.col('old')>0).then(1).otherwise(0).alias('old'))

check_oz = check_oz.with_columns(pl.when(pl.col('обеспечение заявки').is_not_null()).then(1).otherwise(0).alias('new'))
check_oz = check_oz.drop(['oldr','oldpr'])
check_oz.filter(pl.col('old')!=pl.col('new'))

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее,обеспечение заявки,обеспечение исполнения контракта,"обеспечение исполнения контракта, руб","обеспечение исполнения контракта, %",old,new
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64,f64,str,f64,f64,i32,i32
2,"""АО ""ИЭМЗ ""КУПОЛ""""","""1831083343""",null,"""32110203938""",null,"""[ОКПД2 43.99] Работы строитель…","""Удмуртская Республика""","""Ижевск""",2021-04-19 08:44:15,2021-05-13 08:00:00,2021-04-19 00:00:00,2021-05-27 15:00:00,"""ООО ""СВОБОДА КЛИМАТА - ИЖЕВСК""""","""1832084830""","""Победитель""","""Допущен""",0.071228,"""Торговая процедура""",null,94624.97,null,null,null,0,0,null,null,null,null,1,0
1,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",3.8429e7,"""32110870449""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2021-11-26 14:41:42,2021-12-14 11:15:00,2021-11-26 00:00:00,2021-12-16 13:00:00,"""ООО ""МОНТАЖСТРОЙ""""","""7708378307""","""Победитель""","""Допущен""",0.004,"""Торговая процедура""","""Конкурс открытый""",1.9214e6,null,null,null,0,0,null,null,null,null,1,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32109986927""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2021-02-12 12:07:14,2021-03-01 10:15:00,2021-02-12 00:00:00,2021-03-03 11:00:00,"""ООО ""ССР""""","""7729790409""",null,"""Допущен""",0.006,"""Торговая процедура""",null,480144.13,null,null,null,0,0,null,null,null,null,1,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32008978896""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2020-03-12 13:32:57,2020-03-30 10:30:00,2020-03-12 00:00:00,2020-04-03 10:00:00,"""АО ""ГК ""ЕКС""""","""5012000639""","""Победитель""","""Допущен""",0.024523,"""Торговая процедура""",null,271649.4,null,null,null,0,0,null,null,null,null,1,0
1,"""МУП АГО ""АНГАРСКИЙ ВОДОКАНАЛ""""","""3801006828""",659411.44,"""32312441895""",null,"""[ОКПД2 43.29] Работы строитель…","""Иркутская область""","""Иркутская область""",2023-05-31 11:19:38,2023-06-08 05:00:00,2023-05-30 19:00:00,2023-06-12 19:00:00,"""ООО ""АНГАРАСТРОЙ""""","""3801058343""","""Победитель""","""Допущен""",0.167439,"""Торговая процедура""","""Запрос цен""",32970.57,null,null,null,0,0,null,null,null,null,1,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,"""АО ""ВОЭ""""","""3443029580""",1.2291e7,"""32009394462""",null,"""[ОКПД2 41.20] Здания и работы …","""Волгоградская область""","""Волгоградская область""",2020-08-11 14:17:20,2020-08-18 10:00:00,2020-08-11 00:00:00,2020-09-08 12:00:00,"""ООО ""ЭНЕРГОМАШСЕРВИС""""","""3435026659""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Запрос предложений в электронн…",245813.31,null,null,null,0,0,null,null,null,null,1,0
1,"""АО ""ОЗК""""","""7708632345""",1.4832e8,"""32211540637""",null,"""[ОКПД2 52.10] Услуги по склади…","""Москва""","""Москва""",2022-07-11 18:17:41,2022-08-09 10:00:00,2022-07-11 00:00:00,2022-08-23 13:30:00,"""АО ""РАМЕНСКИЙ КОМБИНАТ ХЛЕБОПР…","""5040009908""","""Победитель""","""Допущен""",0.029126,"""Торговая процедура""","""Конкурс открытый с ограниченны…",5000.0,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,20

все максимально печально

In [103]:
check_oz.filter(pl.col('Реестровый номер публикации')=='32110158113')

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее,обеспечение заявки,обеспечение исполнения контракта,"обеспечение исполнения контракта, руб","обеспечение исполнения контракта, %",old,new
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64,f64,str,f64,f64,i32,i32
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""АО ""ФРЕЙТ ЛИНК""""","""7728142525""",null,"""Допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""АО ""ПОЧТА РОССИИ""""","""7724490000""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""АО ""ДПД РУС""""","""7713215523""",null,"""Допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""ООО ""СИТИ РАПИД""""","""7718273254""",null,"""Допущен""",0.993855,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""ООО ""КСЭ""""","""7723720638""",null,"""Допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""АО ""ДХЛ ИНТЕРНЕШНЛ""""","""7707033437""",null,"""Не допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""ФГУП ГЦСС""","""7717043113""",null,"""Допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0,null,null,null,null,1,0


## Заполним пропуски

In [92]:
data = pl.read_excel(source='correct_data/correct_data_2020.xlsx')

In [93]:
data.columns

['Уровень',
 'Заказчик',
 'ИНН заказчика',
 'Стоимость(руб.) Заказчик',
 'Реестровый номер публикации',
 'Идентификационный код закупки',
 'Сфера деятельности',
 'Регион поставки',
 'Город поставки',
 'Дата публикации',
 'Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ',
 'Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ',
 'Дата окончания проведения торгов',
 'Поставщик',
 'ИНН поставщика',
 'Победитель',
 'Статус допуска',
 'Снижение на торгах,%',
 'Форма публикации',
 'Тип торгов',
 'Обеспечение заявки (руб.)',
 'Обеспечение заявки, %',
 'Обеспечение контракта (руб.)',
 'Обеспечение контракта, %',
 'РНП сейчас',
 'РНП ранее']

In [94]:
data.filter(pl.col('Стоимость(руб.) Заказчик').is_not_null() & 
            pl.col('Обеспечение заявки (руб.)').is_not_null() &
            pl.col('Обеспечение заявки, %').is_null())

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
1,"""ОГКУ ""ЦЕНТР СОЦИАЛЬНОЙ ПОДДЕРЖ…","""3906147174""",3.0000e6,"""0135200000520000001""","""20-23906147174390601001-0002-0…","""[ОКПД2 86.90] Услуги в области…","""Калининградская область""","""Калининград""",2020-01-20 11:24:06,2020-01-29 11:00:00,2020-01-20 12:26:14,2020-01-30 00:00:00,"""ГБСУСО КО ""СВЕТЛОГОРСКИЙ СОЦИА…","""3912006140""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Аукцион электронный""",29999.99,null,149999.93,null,0,0
1,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",3.8429e7,"""32110870449""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2021-11-26 14:41:42,2021-12-14 11:15:00,2021-11-26 00:00:00,2021-12-16 13:00:00,"""ООО ""МОНТАЖСТРОЙ""""","""7708378307""","""Победитель""","""Допущен""",0.004,"""Торговая процедура""","""Конкурс открытый""",1.9214e6,null,null,null,0,0
1,"""АДМИНИСТРАЦИЯ ЗНАМЕНСКОГО МУНИ…","""5513001802""",1.541188e6,"""0152300025420000001""","""20-35513001802551301001-0004-0…","""[ОКПД2 49.31] Услуги по перево…","""Омская область""","""Знаменский район""",2020-01-09 19:43:09,2020-01-17 09:00:00,2020-01-09 19:43:09,2020-01-20 00:00:00,"""АО ""ОМСКОБЛАВТОТРАНС""""","""5507249611""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Аукцион электронный""",7705.94,null,77059.4,null,0,0
1,"""МУП АГО ""АНГАРСКИЙ ВОДОКАНАЛ""""","""3801006828""",659411.44,"""32312441895""",null,"""[ОКПД2 43.29] Работы строитель…","""Иркутская область""","""Иркутская область""",2023-05-31 11:19:38,2023-06-08 05:00:00,2023-05-30 19:00:00,2023-06-12 19:00:00,"""ООО ""АНГАРАСТРОЙ""""","""3801058343""","""Победитель""","""Допущен""",0.167439,"""Торговая процедура""","""Запрос цен""",32970.57,null,null,null,0,0
1,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",1.7820e7,"""32110404517""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2021-06-22 12:42:46,2021-07-08 11:15:00,2021-06-22 00:00:00,2021-07-12 13:00:00,"""АО ""ГК ""ЕКС""""","""5012000639""","""Победитель""","""Допущен""",0.001,"""Торговая процедура""","""Конкурс открытый""",890983.9,null,null,null,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
1,"""ГУ-ТОМСКОЕ РО ФОНДА СОЦИАЛЬНОГ…","""7018003855""",259529.78,"""0265100000420000016""","""20-17018003855701701001-0026-0…","""[ОКПД2 53.10] Услуги почтовой …","""Томская область""","""Томская область""",2020-01-15 11:50:12,2020-01-24 09:00:00,2020-01-15 11:51:58,2020-01-27 00:00:00,"""ООО ""РЕДАКЦИЯ ИЗДАНИЯ ""СОЦИАЛЬ…","""7710341840""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Аукцион электронный""",2595.3,null,12976.49,null,0,0
1,"""АДМИНИСТРАЦИЯ МР ""МИРНИНСКИЙ Р…","""1433017567""",3.9083e6,"""0116300000220000014""","""20-31433017567143301001-0101-0…","""[ОКПД2 49.39] Услуги сухопутно…","""Республика Саха (Якутия)""","""Мирный""",2020-01-21 15:03:06,2020-01-30 12:00:00,2020-01-21 15:27:09,2020-01-31 00:00:00,"""МУП ""ЧАРОИТ""""","""1433014799""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Аукцион электронный""",39083.14,null,195415.71,null,0,0
1,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",2.0783e7,"""32110501225""",null,"""[ОКПД2 36.00] Вода природная; …","""Москва""","""Москва""",2021-07-26 15:08:40,2021-08-11 11:15:00,2021-07-26 00:00:0

In [101]:
data.filter(pl.col('Стоимость(руб.) Заказчик').is_null() & 
            pl.col('Обеспечение заявки (руб.)').is_not_null())

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
2,"""АО ""ИЭМЗ ""КУПОЛ""""","""1831083343""",null,"""32110203938""",null,"""[ОКПД2 43.99] Работы строитель…","""Удмуртская Республика""","""Ижевск""",2021-04-19 08:44:15,2021-05-13 08:00:00,2021-04-19 00:00:00,2021-05-27 15:00:00,"""ООО ""СВОБОДА КЛИМАТА - ИЖЕВСК""""","""1832084830""","""Победитель""","""Допущен""",0.071228,"""Торговая процедура""",null,94624.97,null,null,null,0,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32109986927""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2021-02-12 12:07:14,2021-03-01 10:15:00,2021-02-12 00:00:00,2021-03-03 11:00:00,"""ООО ""ССР""""","""7729790409""",null,"""Допущен""",0.006,"""Торговая процедура""",null,480144.13,null,null,null,0,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32008978896""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2020-03-12 13:32:57,2020-03-30 10:30:00,2020-03-12 00:00:00,2020-04-03 10:00:00,"""АО ""ГК ""ЕКС""""","""5012000639""","""Победитель""","""Допущен""",0.024523,"""Торговая процедура""",null,271649.4,null,null,null,0,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32009036628""",null,"""[ОКПД2 71.12] Услуги в области…","""Москва""","""Москва""",2020-03-26 15:31:43,2020-04-13 11:15:00,2020-03-26 00:00:00,2020-04-20 10:00:00,"""ООО ""ГРАДПРОЕКТКОМПАНИ""""","""7723827740""",null,"""Не допущен""",0.0,"""Торговая процедура""",null,608746.41,null,null,null,0,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32110177974""",null,"""[ОКПД2 43.99] Работы строитель…","""Москва""","""Москва""",2021-04-09 14:57:11,2021-04-26 10:30:00,2021-04-09 00:00:00,2021-04-30 10:00:00,"""ООО ""СИСТЕМА""""","""5032257329""","""Победитель""","""Допущен""",0.01,"""Торговая процедура""",null,662984.08,null,null,null,0,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,"""МУП ""ВОДОКАНАЛ""""","""6608001915""",null,"""32110164274""",null,"""[ОКПД2 71.12] Услуги в области…","""Свердловская область""","""Свердловская область""",2021-04-06 17:34:59,2021-04-22 10:00:00,2021-04-06 00:00:00,2021-04-27 00:00:00,"""ООО ""ППК ГРАД""""","""6670021590""","""Победитель""","""Допущен""",0.183333,"""Торговая процедура""",null,207855.64,null,null,null,0,0
2,"""АО ""МОСВОДОКАНАЛ""""","""7701984274""",null,"""32211107741""",null,"""[ОКПД2 42.21] Сооружения и стр…","""Москва""","""Москва""",2022-02-07 14:50:56,2022-02-24 10:30:00,2022-02-07 00:00:00,2022-02-28 13:00:00,"""ООО ""ИНЖСТРОЙ-ИННОВАЦИИ""""","""5050132940""","""Победитель""","""Допущен""",0.002,"""Торговая процедура""",null,1.5837e6,null,null,null,0,0
2,"""БАНК ВТБ (ПАО)""","""7702070139""",null,"""32110158113""",null,"""[ОКПД2 53.10] Услуги почтовой …","""Санкт-Петербург""","""Санкт-Петербург""",2021-04-05 10:47:35,2021-04-16 12:00:00,2021-04-05 00:00:00,2021-05-14 10:00:00,"""ФГУП ГЦСС""","""7717043113""",null,"""Допущен""",0.0,"""Торговая процедура""",null,5.34e6,null,null,null,0,0


In [104]:
data.null_count()

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,36823,0,76143,14,0,0,0,0,0,0,0,0,44374,0,0,0,66199,91380,96972,95890,79832,0,0


In [95]:
new = data.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_not_null() &
                                pl.col('Обеспечение заявки (руб.)').is_not_null() &
                                pl.col('Обеспечение заявки, %').is_null())
                            .then(pl.col('Обеспечение заявки (руб.)')/pl.col('Стоимость(руб.) Заказчик'))
                            .otherwise(pl.col('Обеспечение заявки, %'))
                            .alias('Обеспечение заявки, %'))
new = new.with_columns(pl.when(pl.col('Стоимость(руб.) Заказчик').is_not_null() &
                                pl.col('Обеспечение заявки (руб.)').is_null() &
                                pl.col('Обеспечение заявки, %').is_not_null())
                            .then(pl.col('Обеспечение заявки, %')*pl.col('Стоимость(руб.) Заказчик'))
                            .otherwise(pl.col('Обеспечение заявки (руб.)'))
                            .alias('Обеспечение заявки (руб.)'))


In [97]:
new.filter(pl.col('Реестровый номер публикации')=='32009394462')

Уровень,Заказчик,ИНН заказчика,Стоимость(руб.) Заказчик,Реестровый номер публикации,Идентификационный код закупки,Сфера деятельности,Регион поставки,Город поставки,Дата публикации,Дата окончания приема заявок / Дата планового окончания исполнения контракта / Плановая дата публикации лота по ППГ,Дата начала подачи заявок / Дата начала исполнения контракта / Дата публикации ППГ,Дата окончания проведения торгов,Поставщик,ИНН поставщика,Победитель,Статус допуска,"Снижение на торгах,%",Форма публикации,Тип торгов,Обеспечение заявки (руб.),"Обеспечение заявки, %",Обеспечение контракта (руб.),"Обеспечение контракта, %",РНП сейчас,РНП ранее
i64,str,str,f64,str,str,str,str,str,datetime[ms],datetime[ms],datetime[ms],datetime[ms],str,str,str,str,f64,str,str,f64,f64,f64,f64,i64,i64
1,"""АО ""ВОЭ""""","""3443029580""",1.2291e7,"""32009394462""",null,"""[ОКПД2 41.20] Здания и работы …","""Волгоградская область""","""Волгоградская область""",2020-08-11 14:17:20,2020-08-18 10:00:00,2020-08-11 00:00:00,2020-09-08 12:00:00,"""ООО ""ЭНЕРГОМАШСЕРВИС""""","""3435026659""","""Победитель""","""Допущен""",0.0,"""Торговая процедура""","""Запрос предложений в электронн…",245813.31,0.02,null,null,0,0


In [170]:
df.loc[291:294]

,inn,price,num
291,223003534,124010.00,301300403220000000
292,224003713,491872.08,301300001120000000
293,224003713,665000.00,301300001120000000
294,224003713,850000.00,301300001120000000
